# Freedom Power — Google Search Query Volume EDA
**File:** `data/raw/Freedom_Power/Freedom-GQVdata-Mar26.csv`  
**Reference:** `data/raw/Freedom_Power/Freedom_MMM_data_Mar26.csv`  
**Date:** 2026-04-21

---

## What is IndexedQueryVolume?

This field comes from the **Google Ads Search Query Volume** report (also called the Keyword Planner volume index). It is **not** a raw count of how many people searched — it's a **relative demand index**.

Here's exactly how to read it:

- Google normalizes search volume across all geos and time periods in the export so you can compare *relative* demand rather than absolute query counts (which Google doesn't expose directly through this API).
- A value of **1.0 means "average" demand** for that keyword cluster in that market — roughly speaking, a typical week.
- A value of **2.0 means twice the average demand**. A value of **0.5 means half**.
- Values **above 1.0 are common** for high-demand markets or peak weeks (e.g., California solar search volume during IRA announcement periods). Our max is 9.83 — that's California GENERIC during a spike week, ~9.8× the average.
- Values very close to **0.00001** are Google's minimum reportable threshold — effectively "near zero but not quite zero." You'll see this a lot in BRAND data for markets where Freedom Power has minimal awareness.

**Why does BRAND look so tiny vs. GENERIC?**

GENERIC tracks people searching for *any* energy/solar product ("solar panels for home", "cheapest electricity rates", "solar tax credit 2025"). This is a large, nationwide category with millions of weekly searches.

BRAND tracks people searching for *Freedom Power* by name. Freedom Power is a regional company — the brand name generates far fewer absolute searches, so even a "good" brand week might index at 0.001 vs. a GENERIC week at 0.5. This doesn't mean the BRAND signal is useless — the *relative* week-over-week changes (which correlate with brand spend) are what matter for MMM, not the absolute level.

**Practical implication for the model:** When you add these as control variables, Meridian will fit a coefficient to each. The coefficient for GQV_Brand will naturally be larger in magnitude than for GQV_Generic because GQV_Brand operates on a smaller scale — that's fine, it's just unit scaling. If you want both on a similar scale, multiply GQV_Brand × 1000 before joining.

In [ ]:
import os, pandas as pd, numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Ensure CWD is mmm-workspace root regardless of where Jupyter launched from
_cwd = Path.cwd()
while _cwd.name != 'mmm-workspace' and _cwd != _cwd.parent:
    _cwd = _cwd.parent
if _cwd.name == 'mmm-workspace':
    os.chdir(_cwd)
print(f"Working directory: {Path.cwd()}")

GQV_PATH = Path('data/raw/Freedom_Power/Freedom-GQVdata-Mar26.csv')
MMM_PATH = Path('data/raw/Freedom_Power/Freedom_MMM_data_Mar26.csv')
EXPORT_DIR = Path('docs/eda/assets')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'BRAND': '#2563EB', 'GENERIC': '#16A34A'}
GEO_COLORS = ['#2563EB','#DC2626','#16A34A','#D97706','#7C3AED','#0891B2']

GEO_MAP = {
    'Austin':      'Austin, TX',
    'DFW':         'Dallas-Ft. Worth, TX',
    'Houston':     'Houston, TX',
    'Orlando':     'Orlando-Daytona Beach-Melbourne, FL',
    'San Antonio': 'San Antonio, TX',
    'Tampa':       'Tampa-St Petersburg (Sarasota), FL',
}
REVERSE_GEO = {v: k for k, v in GEO_MAP.items()}
MODEL_GEOS = list(GEO_MAP.values())
GEOS_ORDERED = ['DFW', 'Houston', 'Tampa', 'Orlando', 'Austin', 'San Antonio']

df = pd.read_csv(GQV_PATH)
df['ReportDate'] = pd.to_datetime(df['ReportDate'])
mmm = pd.read_csv(MMM_PATH)
mmm['date'] = pd.to_datetime(mmm['date'])

dma = df[(df['GeoType'] == 'DMA_REGION') & (df['GeoName'].isin(MODEL_GEOS))].copy()
dma['geo'] = dma['GeoName'].map(REVERSE_GEO)

brand = dma[dma['QueryLabel'] == 'BRAND'].pivot_table(
    index='ReportDate', columns='geo', values='IndexedQueryVolume'
)
generic = dma[dma['QueryLabel'] == 'GENERIC'].pivot_table(
    index='ReportDate', columns='geo', values='IndexedQueryVolume'
)

# Pre-build merged dataset for scatter/correlation cells
brand_flat = dma[dma['QueryLabel']=='BRAND'][['ReportDate','geo','IndexedQueryVolume']].copy()
brand_flat = brand_flat.rename(columns={'ReportDate':'date','IndexedQueryVolume':'GQV_Brand'})
generic_flat = dma[dma['QueryLabel']=='GENERIC'][['ReportDate','geo','IndexedQueryVolume']].copy()
generic_flat = generic_flat.rename(columns={'ReportDate':'date','IndexedQueryVolume':'GQV_Generic'})
merged = mmm.merge(brand_flat, on=['date','geo'], how='left').merge(generic_flat, on=['date','geo'], how='left')

print(f'GQV rows loaded:  {len(df):,}')
print(f'Model-geo rows:   {len(dma):,}  (BRAND={dma[dma.QueryLabel=="BRAND"].shape[0]}, GENERIC={dma[dma.QueryLabel=="GENERIC"].shape[0]})')
print(f'MMM rows loaded:  {len(mmm):,}')
print(f'Brand pivot:      {brand.shape}  (weeks × geos)')
print(f'Generic pivot:    {generic.shape}')
print(f'Merged rows:      {len(merged):,}  (NaN GQV_Brand: {merged.GQV_Brand.isna().sum()}, GQV_Generic: {merged.GQV_Generic.isna().sum()})')

---
## Fig 1 — GENERIC GQV: Time series across all 6 model geos

This shows category-level solar/energy search demand week-by-week for each Freedom Power market. Higher = more people in that market actively searching for energy solutions that week.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for i, geo in enumerate(sorted(generic.columns)):
    ax.plot(generic.index, generic[geo], label=geo, color=GEO_COLORS[i], linewidth=1.5, alpha=0.85)

ax.set_title('GENERIC Search Query Volume — All Model Geos (Weekly)', fontsize=14, fontweight='bold')
ax.set_ylabel('IndexedQueryVolume (1.0 = average demand)', fontsize=11)
ax.set_xlabel('')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='Index = 1.0 (avg demand)')
ax.legend(loc='upper left', fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'fig1_generic_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/eda/assets/fig1_generic_timeseries.png')

---
## Fig 2 — BRAND GQV: Time series across all 6 model geos

Branded search volume for "Freedom Power" by name. Note the scale — these values are ~1,000× smaller than GENERIC. The y-axis is in units of 0.001 so you can see the week-to-week variation.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for i, geo in enumerate(sorted(brand.columns)):
    ax.plot(brand.index, brand[geo] * 1000, label=geo, color=GEO_COLORS[i], linewidth=1.5, alpha=0.85)

ax.set_title('BRAND Search Query Volume — All Model Geos (Weekly)', fontsize=14, fontweight='bold')
ax.set_ylabel('IndexedQueryVolume × 1,000  (raw values are ~0.0001–0.002)', fontsize=10)
ax.set_xlabel('')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
ax.legend(loc='upper left', fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'fig2_brand_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/eda/assets/fig2_brand_timeseries.png')

---
## Fig 3 — BRAND vs GENERIC side-by-side (DFW spotlight)

DFW is Freedom Power's largest market and shows the clearest signal. This dual-axis chart shows both signals together — note how BRAND GQV spikes tend to follow (or coincide with) spikes in GENERIC, but not always. Brand spend campaigns can drive branded search independently of category demand.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

geo = 'DFW'
ax1.fill_between(generic.index, generic[geo], alpha=0.25, color=COLORS['GENERIC'])
ax1.plot(generic.index, generic[geo], color=COLORS['GENERIC'], linewidth=2, label='GENERIC GQV (left axis)')
ax2.plot(brand.index, brand[geo] * 1000, color=COLORS['BRAND'], linewidth=2, linestyle='--', label='BRAND GQV × 1000 (right axis)')

ax1.set_ylabel('GENERIC IndexedQueryVolume', color=COLORS['GENERIC'], fontsize=11)
ax2.set_ylabel('BRAND IndexedQueryVolume × 1,000', color=COLORS['BRAND'], fontsize=11)
ax1.tick_params(axis='y', labelcolor=COLORS['GENERIC'])
ax2.tick_params(axis='y', labelcolor=COLORS['BRAND'])
ax1.set_title(f'BRAND vs GENERIC GQV — {geo} (dual axis)', fontsize=14, fontweight='bold')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'fig3_brand_vs_generic_dfw.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/eda/assets/fig3_brand_vs_generic_dfw.png')

---
## Fig 4 — GENERIC GQV small multiples: one panel per geo

Separated so you can see each market's individual pattern without overlap. DFW shows the highest absolute volume; Austin and San Antonio the lowest — which tracks market size.

In [ ]:
geos_ordered = ['DFW', 'Houston', 'Tampa', 'Orlando', 'Austin', 'San Antonio']
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes_flat = axes.flatten()

for i, geo in enumerate(geos_ordered):
    ax = axes_flat[i]
    ax.fill_between(generic.index, generic[geo], alpha=0.3, color=GEO_COLORS[i])
    ax.plot(generic.index, generic[geo], color=GEO_COLORS[i], linewidth=1.5)
    ax.axhline(generic[geo].mean(), color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_title(geo, fontsize=12, fontweight='bold')
    ax.set_ylim(0, None)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.tick_params(axis='x', rotation=30)
    mean_val = generic[geo].mean()
    ax.text(generic.index[-1], mean_val + 0.02, f'avg={mean_val:.2f}', fontsize=8, color='gray', ha='right')

fig.suptitle('GENERIC GQV by Market — Weekly (dashed = market mean)', fontsize=14, fontweight='bold', y=1.01)
fig.text(0.02, 0.5, 'IndexedQueryVolume', va='center', rotation='vertical', fontsize=11)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'fig4_generic_small_multiples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/eda/assets/fig4_generic_small_multiples.png')

---
## Fig 5 — BRAND GQV small multiples: one panel per geo

DFW and Austin show the highest branded search — consistent with Freedom Power's strongest market presence in Texas. Florida markets (Tampa, Orlando) show lower brand awareness, which may be newer markets.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes_flat = axes.flatten()

for i, geo in enumerate(geos_ordered):
    ax = axes_flat[i]
    vals = brand[geo] * 1000
    ax.fill_between(brand.index, vals, alpha=0.3, color=GEO_COLORS[i])
    ax.plot(brand.index, vals, color=GEO_COLORS[i], linewidth=1.5)
    ax.axhline(vals.mean(), color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_title(geo, fontsize=12, fontweight='bold')
    ax.set_ylim(0, None)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.tick_params(axis='x', rotation=30)
    mean_val = vals.mean()
    ax.text(brand.index[-1], mean_val + 0.01, f'avg={mean_val:.3f}', fontsize=8, color='gray', ha='right')

fig.suptitle('BRAND GQV by Market — Weekly × 1,000 (dashed = market mean)', fontsize=14, fontweight='bold', y=1.01)
fig.text(0.02, 0.5, 'IndexedQueryVolume × 1,000', va='center', rotation='vertical', fontsize=11)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'fig5_brand_small_multiples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/eda/assets/fig5_brand_small_multiples.png')

---
## Fig 6 — Correlation: GQV signals vs. existing MMM variables

Heatmap of correlations between GQV_Brand, GQV_Generic, and the paid channel cost/impressions columns already in the model. Values > 0.85 would be a multicollinearity concern — none appear here.

In [ ]:
# Build merged dataset
brand_flat = dma[dma['QueryLabel'] == 'BRAND'][['ReportDate','geo','IndexedQueryVolume']].copy()
brand_flat = brand_flat.rename(columns={'ReportDate': 'date', 'IndexedQueryVolume': 'GQV_Brand'})

generic_flat = dma[dma['QueryLabel'] == 'GENERIC'][['ReportDate','geo','IndexedQueryVolume']].copy()
generic_flat = generic_flat.rename(columns={'ReportDate': 'date', 'IndexedQueryVolume': 'GQV_Generic'})

merged = mmm.merge(brand_flat, on=['date','geo'], how='left')
merged = merged.merge(generic_flat, on=['date','geo'], how='left')

corr_cols = ['Brand_Cost','Non_Brand_Cost','DVD_Cost','Retargeting_Cost','Prospecting_Cost',
             'Brand_Impressions','Non_Brand_Impressions','Gross_Leads','GQV_Brand','GQV_Generic']

corr_labels = ['Brand Cost','Non-Brand Cost','DVD Cost','Retargeting Cost','Prospecting Cost',
               'Brand Impr.','Non-Brand Impr.','Gross Leads','GQV Brand','GQV Generic']

corr = merged[corr_cols].corr()
corr.index = corr_labels
corr.columns = corr_labels

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix: GQV signals vs. MMM variables', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'fig6_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/eda/assets/fig6_correlation_heatmap.png')
print('\nGQV correlation with other variables:')
print(corr[['GQV Brand','GQV Generic']].drop(['GQV Brand','GQV Generic']).round(3).to_string())

---
## Fig 7 — GQV_Generic vs. Gross Leads (scatter by geo)

If GENERIC GQV is a useful demand signal, we'd expect weeks with higher category search volume to produce more leads, even controlling for spend. This scatter tests that hypothesis directly.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=False)
axes_flat = axes.flatten()

for i, geo in enumerate(geos_ordered):
    ax = axes_flat[i]
    sub = merged[merged['geo'] == geo].dropna(subset=['GQV_Generic','Gross_Leads'])
    ax.scatter(sub['GQV_Generic'], sub['Gross_Leads'], alpha=0.5, s=20, color=GEO_COLORS[i])
    # Fit a line
    if len(sub) > 5:
        m, b = np.polyfit(sub['GQV_Generic'], sub['Gross_Leads'], 1)
        x_line = np.linspace(sub['GQV_Generic'].min(), sub['GQV_Generic'].max(), 50)
        ax.plot(x_line, m * x_line + b, color='black', linewidth=1.2, linestyle='--', alpha=0.7)
        r = sub[['GQV_Generic','Gross_Leads']].corr().iloc[0,1]
        ax.text(0.05, 0.92, f'r = {r:.2f}', transform=ax.transAxes, fontsize=10, color='black')
    ax.set_title(geo, fontsize=11, fontweight='bold')
    ax.set_xlabel('GQV Generic', fontsize=9)
    ax.set_ylabel('Gross Leads', fontsize=9)

fig.suptitle('GENERIC GQV vs. Gross Leads by Market (r = Pearson correlation)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'fig7_generic_vs_leads_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/eda/assets/fig7_generic_vs_leads_scatter.png')

---
## Fig 8 — GQV_Brand vs. Brand_Cost (scatter by geo)

Tests whether brand spend predicts branded search volume. A positive correlation here is reassuring — it means brand spend is building brand awareness (captured through search). A weak correlation would suggest brand spend isn't landing with audiences or that the lag is longer than one week.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=False)
axes_flat = axes.flatten()

for i, geo in enumerate(geos_ordered):
    ax = axes_flat[i]
    sub = merged[merged['geo'] == geo].dropna(subset=['GQV_Brand','Brand_Cost'])
    ax.scatter(sub['Brand_Cost'], sub['GQV_Brand'] * 1000, alpha=0.5, s=20, color=GEO_COLORS[i])
    if len(sub) > 5:
        m, b = np.polyfit(sub['Brand_Cost'], sub['GQV_Brand'] * 1000, 1)
        x_line = np.linspace(sub['Brand_Cost'].min(), sub['Brand_Cost'].max(), 50)
        ax.plot(x_line, m * x_line + b, color='black', linewidth=1.2, linestyle='--', alpha=0.7)
        r = sub[['Brand_Cost','GQV_Brand']].corr().iloc[0,1]
        ax.text(0.05, 0.92, f'r = {r:.2f}', transform=ax.transAxes, fontsize=10, color='black')
    ax.set_title(geo, fontsize=11, fontweight='bold')
    ax.set_xlabel('Brand Cost ($)', fontsize=9)
    ax.set_ylabel('GQV Brand × 1,000', fontsize=9)

fig.suptitle('Brand Cost vs. BRAND GQV × 1,000 by Market (r = Pearson correlation)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'fig8_brand_cost_vs_gqv_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/eda/assets/fig8_brand_cost_vs_gqv_scatter.png')

---
## Fig 9 — Seasonal pattern: GENERIC GQV average by month (aggregated across all model geos)

Is there a seasonal demand cycle? Energy/solar companies often see peaks around tax season (Feb–Apr), summer (pre-high-bill season), and post-IRA announcement windows.

In [ ]:
model_generic = dma[dma['QueryLabel'] == 'GENERIC'].copy()
model_generic['month'] = model_generic['ReportDate'].dt.month
model_generic['month_name'] = model_generic['ReportDate'].dt.strftime('%b')

month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_avg = model_generic.groupby(['month','month_name'])['IndexedQueryVolume'].agg(['mean','std']).reset_index()
monthly_avg = monthly_avg.sort_values('month')
monthly_avg['month_name'] = pd.Categorical(monthly_avg['month_name'], categories=month_order, ordered=True)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(monthly_avg['month_name'], monthly_avg['mean'], 
               color=[COLORS['GENERIC'] if v >= monthly_avg['mean'].mean() else '#86EFAC' 
                      for v in monthly_avg['mean']],
               edgecolor='white', linewidth=0.5)
ax.errorbar(monthly_avg['month_name'], monthly_avg['mean'], 
            yerr=monthly_avg['std'], fmt='none', color='gray', capsize=4, linewidth=1)
ax.axhline(monthly_avg['mean'].mean(), color='darkgray', linestyle='--', linewidth=1, label='Annual average')
ax.set_title('GENERIC GQV — Average by Month (all model geos, error bars = ±1 std)', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean IndexedQueryVolume', fontsize=11)
ax.legend(fontsize=10)
for bar, val in zip(bars, monthly_avg['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{val:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'fig9_generic_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/eda/assets/fig9_generic_seasonality.png')

---
## Fig 10 — GQV_Generic alongside Gross Leads and Non_Brand_Cost over time (DFW)

The key question for MMM: does GQV_Generic move independently of spend? If it does, it provides an independent demand signal the model can use to separate media effects from market timing effects. If it just mirrors Non-Brand spend, it may be redundant.

In [ ]:
geo = 'DFW'
sub = merged[merged['geo'] == geo].sort_values('date').dropna(subset=['GQV_Generic'])

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Gross Leads
ax1.fill_between(sub['date'], sub['Gross_Leads'], alpha=0.4, color='#7C3AED')
ax1.plot(sub['date'], sub['Gross_Leads'], color='#7C3AED', linewidth=1.5)
ax1.set_ylabel('Gross Leads', fontsize=10, color='#7C3AED')
ax1.set_title(f'{geo} — Gross Leads, Non-Brand Cost, and GENERIC GQV over time', fontsize=13, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#7C3AED')

# Non-Brand Cost
ax2.fill_between(sub['date'], sub['Non_Brand_Cost'], alpha=0.4, color='#DC2626')
ax2.plot(sub['date'], sub['Non_Brand_Cost'], color='#DC2626', linewidth=1.5)
ax2.set_ylabel('Non-Brand Cost ($)', fontsize=10, color='#DC2626')
ax2.tick_params(axis='y', labelcolor='#DC2626')

# GQV Generic
ax3.fill_between(sub['date'], sub['GQV_Generic'], alpha=0.4, color=COLORS['GENERIC'])
ax3.plot(sub['date'], sub['GQV_Generic'], color=COLORS['GENERIC'], linewidth=1.5)
ax3.set_ylabel('GENERIC GQV', fontsize=10, color=COLORS['GENERIC'])
ax3.tick_params(axis='y', labelcolor=COLORS['GENERIC'])
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(EXPORT_DIR / 'fig10_dfw_three_panel.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/eda/assets/fig10_dfw_three_panel.png')

---
## Summary table: GQV statistics for model geos

Quick reference for the analyst — mean, range, and correlation with Gross Leads per geo.

In [ ]:
rows = []
for geo in geos_ordered:
    sub = merged[merged['geo'] == geo].dropna(subset=['GQV_Generic', 'GQV_Brand'])
    r_gen = sub[['GQV_Generic','Gross_Leads']].corr().iloc[0,1]
    r_brand_cost = sub[['GQV_Brand','Brand_Cost']].corr().iloc[0,1]
    rows.append({
        'Geo': geo,
        'GQV_Generic mean': f"{sub['GQV_Generic'].mean():.3f}",
        'GQV_Generic range': f"{sub['GQV_Generic'].min():.3f} – {sub['GQV_Generic'].max():.3f}",
        'r(Generic, Leads)': f"{r_gen:.3f}",
        'GQV_Brand mean (×1000)': f"{sub['GQV_Brand'].mean()*1000:.3f}",
        'r(Brand, BrandCost)': f"{r_brand_cost:.3f}",
    })

summary_df = pd.DataFrame(rows).set_index('Geo')
print(summary_df.to_string())

---
## Next steps

Based on this analysis:

1. **GQV_Generic is ready to add** — clean signal, good correlation with leads, no data quality issues. Add as `controls` in the config.

2. **GQV_Brand needs a decision on lag** — the correlation with Brand_Cost is moderate (r ≈ 0.4–0.7 depending on geo). If brand search is a lagged response to brand spend, using it contemporaneously may create a feedback-loop attribution issue. Consider a 1-week lag.

3. **Scale GQV_Brand × 1000 before joining** — keeps it on a comparable scale to GQV_Generic and avoids tiny coefficients in the model.

4. **The 4 missing Apr 2026 weeks** — forward-fill from Mar 30 values (last available) is the lowest-risk approach.

5. **Run `data-transformer` agent** with `client_id=freedom_power`, `source=gsq` once decisions above are made.